# Source Injection into Rubin Images

**Inject custom star cluster profiles into Rubin calexp/deepCoadd images**

- Uses our `src.light_profiles` (Plummer, King, EFF, Sersic) for cluster models
- Uses **GalSim** for PSF modeling and convolution
- Fool-proof: no coadd PSF extraction that can fail

Butler pattern from: [DP02_14](https://github.com/rubin-dp0/tutorial-notebooks/blob/main/DP02_14_Injecting_Synthetic_Sources.ipynb)

In [ ]:
# ==============================================================================
# IMPORTS
# ==============================================================================
import os
import sys
import inspect
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Circle
from collections import Counter

# GalSim for PSF
import galsim

# Rubin Science Pipelines
from lsst.daf.butler import Butler
from lsst.geom import Point2D

# ==============================================================================
# ADD OUR PIPELINE
# ==============================================================================
USERNAME = os.environ.get('USER', 'your_username')
PIPELINE_PATH = f'/home/{USERNAME}/INJECT/star-cluster-injection-pipeline'

# Fallback paths to try
FALLBACK_PATHS = [
    PIPELINE_PATH,
    os.path.expanduser('~/INJECT/star-cluster-injection-pipeline'),
    '/Users/snehanair/Documents/GitHub/INJECT/star-cluster-injection-pipeline',
    os.path.join(os.path.dirname(os.getcwd()), ''),  # parent of notebooks
]

pipeline_found = False
for path in FALLBACK_PATHS:
    if os.path.exists(path) and os.path.isdir(path):
        if path not in sys.path:
            sys.path.insert(0, path)
        PIPELINE_PATH = path
        pipeline_found = True
        print(f'Pipeline found: {path}')
        break

if not pipeline_found:
    raise FileNotFoundError(
        f'Could not find pipeline. Tried: {FALLBACK_PATHS}\n'
        f'Please set PIPELINE_PATH manually.'
    )

from src.light_profiles import (
    PlummerProfile, KingProfile, EFFProfile, SersicProfile, mag_to_flux
)

plt.style.use('tableau-colorblind10')
print('All imports OK')
print(f'GalSim version: {galsim.__version__}')

In [ ]:
# ==============================================================================
# INSPECT PROFILE SIGNATURES
# ==============================================================================
print('=== Profile constructor signatures ===')
for cls in [PlummerProfile, KingProfile, EFFProfile, SersicProfile]:
    sig = inspect.signature(cls.__init__)
    print(f'{cls.__name__:20s} {sig}')

---
## GalSim PSF Helper Functions

Create PSF models using GalSim. This is more accurate than simple Gaussian
and avoids any issues with extracting PSFs from coadds.

In [ ]:
# ==============================================================================
# GALSIM PSF FUNCTIONS
# ==============================================================================

def make_galsim_psf(fwhm_arcsec, pixel_scale, psf_type='moffat', beta=2.5):
    """
    Create a GalSim PSF object.
    
    Parameters
    ----------
    fwhm_arcsec : float
        PSF FWHM in arcseconds
    pixel_scale : float
        Pixel scale in arcsec/pixel
    psf_type : str
        'gaussian' or 'moffat' (Moffat is more realistic for ground-based)
    beta : float
        Moffat beta parameter (only used if psf_type='moffat')
    
    Returns
    -------
    galsim.GSObject
        The PSF object
    """
    if psf_type.lower() == 'gaussian':
        psf = galsim.Gaussian(fwhm=fwhm_arcsec)
    elif psf_type.lower() == 'moffat':
        psf = galsim.Moffat(fwhm=fwhm_arcsec, beta=beta)
    else:
        raise ValueError(f"Unknown psf_type: {psf_type}. Use 'gaussian' or 'moffat'")
    
    return psf


def get_psf_kernel(psf, pixel_scale, stamp_size=None):
    """
    Render a GalSim PSF to a numpy array.
    
    Parameters
    ----------
    psf : galsim.GSObject
        The PSF object
    pixel_scale : float
        Pixel scale in arcsec/pixel
    stamp_size : int, optional
        Size of output array. If None, auto-determined.
    
    Returns
    -------
    ndarray
        Normalized PSF kernel
    """
    if stamp_size is None:
        # Auto-determine size based on PSF FWHM
        fwhm_pix = psf.fwhm / pixel_scale
        stamp_size = int(np.ceil(fwhm_pix * 8)) | 1  # Ensure odd
        stamp_size = max(stamp_size, 21)
    
    psf_image = psf.drawImage(scale=pixel_scale, nx=stamp_size, ny=stamp_size)
    kernel = psf_image.array.copy()
    kernel /= kernel.sum()  # Normalize
    return kernel


print('GalSim PSF functions defined')

---
## Profile Builder and Injection Functions

In [ ]:
# ==============================================================================
# PROFILE BUILDER
# ==============================================================================

def build_profile(profile_type, flux, r_half, extra_params=None):
    """
    Build a light profile object from our pipeline.
    
    Parameters
    ----------
    profile_type : str
        One of 'plummer', 'king', 'eff', 'sersic'
    flux : float
        Total flux of the profile
    r_half : float
        Half-light radius in pixels
    extra_params : dict, optional
        Additional parameters (concentration, gamma, sersic_n)
    
    Returns
    -------
    Profile object from src.light_profiles
    """
    if extra_params is None:
        extra_params = {}
    
    classes = {
        'plummer': PlummerProfile,
        'king':    KingProfile,
        'eff':     EFFProfile,
        'sersic':  SersicProfile,
    }
    
    ptype = profile_type.lower()
    if ptype not in classes:
        raise ValueError(f"Unknown profile_type: {profile_type}. Use one of {list(classes.keys())}")
    
    cls = classes[ptype]
    params = inspect.signature(cls.__init__).parameters
    
    # Detect flux kwarg name
    flux_keys = ['total_flux', 'flux', 'total_counts', 'amplitude']
    flux_key = next((k for k in flux_keys if k in params), None)
    if flux_key is None:
        raise ValueError(f'{cls.__name__} has no recognized flux arg. Params: {list(params.keys())}')
    
    # Detect r_half kwarg name
    rh_keys = ['r_half', 'r_eff', 'half_light_radius', 'r_core']
    rh_key = next((k for k in rh_keys if k in params), None)
    if rh_key is None:
        raise ValueError(f'{cls.__name__} has no recognized r_half arg. Params: {list(params.keys())}')
    
    kwargs = {flux_key: flux, rh_key: r_half}
    
    # Add profile-specific parameters
    if ptype == 'king' and 'concentration' in params:
        kwargs['concentration'] = extra_params.get('concentration', 30.0)
    if ptype == 'eff' and 'gamma' in params:
        kwargs['gamma'] = extra_params.get('gamma', 2.5)
    if ptype == 'sersic':
        sn_keys = ['sersic_n', 'n', 'index']
        sn_key = next((k for k in sn_keys if k in params), None)
        if sn_key:
            kwargs[sn_key] = extra_params.get('sersic_n', 2.0)
    
    return cls(**kwargs)


print('Profile builder defined')

In [ ]:
# ==============================================================================
# INJECTION FUNCTION - USES GALSIM FOR CONVOLUTION
# ==============================================================================

def inject_cluster(image, x, y, profile, psf, pixel_scale, add_poisson=True, rng=None):
    """
    Inject a single cluster profile into an image using GalSim convolution.
    
    Parameters
    ----------
    image : ndarray
        The image to inject into (modified in place)
    x, y : float
        Position in pixels
    profile : Profile object
        From our src.light_profiles
    psf : galsim.GSObject
        The PSF to convolve with
    pixel_scale : float
        Pixel scale in arcsec/pixel
    add_poisson : bool
        Whether to add Poisson noise
    rng : numpy.random.Generator, optional
        Random number generator
    
    Returns
    -------
    dict
        Info about the injection (stamp, flux, etc.)
    """
    if rng is None:
        rng = np.random.default_rng()
    
    ny, nx = image.shape
    ix, iy = int(round(x)), int(round(y))
    
    # Determine stamp size based on profile extent
    r_half = getattr(profile, 'r_half', getattr(profile, 'r_eff', 10.0))
    stamp_size = max(61, int(r_half * 12) | 1)  # Ensure odd
    half = stamp_size // 2
    
    # Generate raw profile stamp using our pipeline
    yg, xg = np.mgrid[:stamp_size, :stamp_size]
    raw_stamp = profile.surface_brightness(xg - half, yg - half)
    
    # Create GalSim InterpolatedImage from our profile
    # This allows us to use GalSim's accurate convolution
    galsim_stamp = galsim.Image(raw_stamp, scale=pixel_scale)
    galsim_profile = galsim.InterpolatedImage(galsim_stamp, normalization='flux')
    
    # Convolve with PSF using GalSim
    convolved = galsim.Convolve([galsim_profile, psf])
    
    # Draw the convolved image
    conv_image = convolved.drawImage(scale=pixel_scale, nx=stamp_size, ny=stamp_size)
    conv_stamp = conv_image.array.copy()
    
    # Add Poisson noise if requested
    if add_poisson:
        pos_mask = conv_stamp > 0
        noisy_stamp = conv_stamp.copy()
        noisy_stamp[pos_mask] = rng.poisson(conv_stamp[pos_mask].astype(float))
    else:
        noisy_stamp = conv_stamp
    
    # Compute bounds for insertion
    y0 = max(0, iy - half)
    y1 = min(ny, iy - half + stamp_size)
    x0 = max(0, ix - half)
    x1 = min(nx, ix - half + stamp_size)
    
    # Stamp slice
    sy0 = y0 - (iy - half)
    sy1 = sy0 + (y1 - y0)
    sx0 = x0 - (ix - half)
    sx1 = sx0 + (x1 - x0)
    
    # Insert into image
    image[y0:y1, x0:x1] += noisy_stamp[sy0:sy1, sx0:sx1]
    
    return {
        'raw_stamp': raw_stamp,
        'conv_stamp': conv_stamp,
        'noisy_stamp': noisy_stamp,
        'flux_injected': float(noisy_stamp.sum()),
        'peak': float(noisy_stamp.max()),
    }


def inject_catalog(image, catalog, psf, pixel_scale, add_poisson=True, seed=None):
    """
    Inject multiple clusters from a catalog into an image.
    
    Parameters
    ----------
    image : ndarray
        Original image (will be copied, not modified)
    catalog : list of dict
        Each entry needs: x, y, magnitude, r_half, flux, profile_type
    psf : galsim.GSObject
        The PSF to use
    pixel_scale : float
        Pixel scale in arcsec/pixel
    add_poisson : bool
        Whether to add Poisson noise
    seed : int, optional
        Random seed
    
    Returns
    -------
    injected_image : ndarray
        Image with injected sources
    info_list : list of dict
        Information about each injection
    """
    rng = np.random.default_rng(seed)
    injected = image.copy().astype(np.float64)
    info_list = []
    
    for entry in catalog:
        try:
            # Build profile from our pipeline
            profile = build_profile(
                entry['profile_type'],
                entry['flux'],
                entry['r_half'],
                entry  # Pass full entry for extra params
            )
            
            # Inject
            result = inject_cluster(
                injected,
                entry['x'], entry['y'],
                profile, psf, pixel_scale,
                add_poisson=add_poisson,
                rng=rng
            )
            
            # Store info
            info = {
                'id': entry.get('id', len(info_list)),
                'x': entry['x'],
                'y': entry['y'],
                'magnitude': entry['magnitude'],
                'r_half': entry['r_half'],
                'profile_type': entry['profile_type'],
                **result
            }
            info_list.append(info)
            
        except Exception as e:
            print(f"Warning: Failed to inject source {entry.get('id', '?')}: {e}")
            continue
    
    return injected.astype(image.dtype), info_list


print('Injection functions defined')

In [ ]:
# ==============================================================================
# CATALOG GENERATOR
# ==============================================================================

def make_cluster_catalog(n_clusters, image_shape, mag_range, r_half_range,
                         zeropoint, edge_buffer=200, seed=42, profile_types=None):
    """
    Generate a random cluster injection catalog.
    
    Parameters
    ----------
    n_clusters : int
        Number of clusters to generate
    image_shape : tuple
        (ny, nx) image dimensions
    mag_range : tuple
        (min_mag, max_mag)
    r_half_range : tuple
        (min_r_half, max_r_half) in pixels
    zeropoint : float
        Photometric zeropoint
    edge_buffer : int
        Buffer from image edges
    seed : int
        Random seed
    profile_types : list, optional
        Profile types to use. Default: all four.
    
    Returns
    -------
    list of dict
        Catalog entries
    """
    rng = np.random.default_rng(seed)
    ny, nx = image_shape
    
    if profile_types is None:
        profile_types = ['plummer', 'king', 'eff', 'sersic']
    
    catalog = []
    for i in range(n_clusters):
        mag = rng.uniform(*mag_range)
        r_half = 10 ** rng.uniform(np.log10(r_half_range[0]), np.log10(r_half_range[1]))
        ptype = rng.choice(profile_types)
        
        entry = {
            'id': i,
            'x': int(rng.integers(edge_buffer, nx - edge_buffer)),
            'y': int(rng.integers(edge_buffer, ny - edge_buffer)),
            'magnitude': float(mag),
            'r_half': float(r_half),
            'flux': float(mag_to_flux(mag, zero_point=zeropoint)),
            'profile_type': ptype,
        }
        
        # Add profile-specific parameters
        if ptype == 'king':
            entry['concentration'] = float(rng.uniform(10, 100))
        if ptype == 'eff':
            entry['gamma'] = float(rng.uniform(2.2, 3.5))
        if ptype == 'sersic':
            entry['sersic_n'] = float(rng.uniform(1.0, 4.0))
        
        catalog.append(entry)
    
    return catalog


print('Catalog generator defined')

---
# Load Rubin Data

In [ ]:
# ==============================================================================
# FIX WORKING DIRECTORY (RSP sometimes starts in deleted directory)
# ==============================================================================
home = os.path.expanduser('~')
try:
    cwd = os.getcwd()
    if not os.path.isdir(cwd):
        raise FileNotFoundError
except (FileNotFoundError, OSError):
    print(f'Working directory invalid! Changing to {home}')
    os.chdir(home)

print(f'Working directory: {os.getcwd()}')

In [ ]:
# ==============================================================================
# CONNECT TO BUTLER
# ==============================================================================
butler = Butler('dp02', collections='2.2i/runs/DP0.2')
print('Butler connected')

In [ ]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================

# Photometric zeropoint (typical for DC2)
ZEROPOINT = 31.4

# PSF parameters (typical Rubin i-band seeing)
PSF_FWHM_ARCSEC = 0.75  # arcseconds
PSF_TYPE = 'moffat'     # 'gaussian' or 'moffat'
PSF_BETA = 2.5          # Moffat beta (only used if moffat)

# Pixel scales
CALEXP_PIXEL_SCALE = 0.2   # arcsec/px (approximate for calexp)
COADD_PIXEL_SCALE = 0.2    # arcsec/px (standard for coadds)

# Injection parameters
N_CLUSTERS = 30
MAG_RANGE = (19.0, 25.0)
R_HALF_RANGE = (3.0, 25.0)  # pixels
EDGE_BUFFER = 200

print('Configuration set')
print(f'  PSF: {PSF_TYPE}, FWHM={PSF_FWHM_ARCSEC}"')
print(f'  Clusters: {N_CLUSTERS}, mag=[{MAG_RANGE[0]}, {MAG_RANGE[1]}]')

---
# Part A: Inject into Calexp

In [ ]:
# ==============================================================================
# LOAD CALEXP
# ==============================================================================
tract = 3828
where = f"instrument='LSSTCam-imSim' AND skymap='DC2' AND tract={tract} AND detector=19 AND band='i'"
calexp_refs = sorted(list(set(butler.registry.queryDatasets('calexp', where=where))))
print(f'Found {len(calexp_refs)} calexps')

if len(calexp_refs) == 0:
    raise RuntimeError('No calexps found! Check your query.')

dataId = calexp_refs[min(5, len(calexp_refs)-1)].dataId
print(f'Using: {dict(dataId)}')

In [ ]:
# ==============================================================================
# GET CALEXP IMAGE
# ==============================================================================
calexp = butler.get('calexp', dataId=dataId)
calexp_image = calexp.image.array.copy()
ny_cal, nx_cal = calexp_image.shape

# Get pixel scale from WCS
wcs = calexp.getWcs()
calexp_pixel_scale = wcs.getPixelScale().asArcseconds()

print(f'Calexp shape: {calexp_image.shape}')
print(f'Pixel scale: {calexp_pixel_scale:.4f} arcsec/px')

In [ ]:
# ==============================================================================
# CREATE GALSIM PSF FOR CALEXP
# ==============================================================================
calexp_psf = make_galsim_psf(PSF_FWHM_ARCSEC, calexp_pixel_scale, psf_type=PSF_TYPE, beta=PSF_BETA)
calexp_psf_fwhm_px = PSF_FWHM_ARCSEC / calexp_pixel_scale

print(f'GalSim PSF created: {PSF_TYPE}')
print(f'  FWHM: {PSF_FWHM_ARCSEC}" = {calexp_psf_fwhm_px:.2f} px')

In [ ]:
# ==============================================================================
# GENERATE CATALOG AND INJECT
# ==============================================================================
calexp_catalog = make_cluster_catalog(
    N_CLUSTERS, calexp_image.shape,
    mag_range=MAG_RANGE, r_half_range=R_HALF_RANGE,
    zeropoint=ZEROPOINT, edge_buffer=EDGE_BUFFER, seed=42,
)

print(f'Catalog: {len(calexp_catalog)} clusters')
print('Injecting...')

calexp_injected, calexp_info = inject_catalog(
    calexp_image, calexp_catalog, calexp_psf, calexp_pixel_scale,
    add_poisson=True, seed=42
)

print(f'Successfully injected {len(calexp_info)} clusters')

## Calexp Plots

In [ ]:
# ==============================================================================
# PLOT: BEFORE / AFTER / DIFFERENCE
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
v1, v2 = np.percentile(calexp_image, [1, 99])

axes[0].imshow(calexp_image, cmap='gray', origin='lower', vmin=v1, vmax=v2)
axes[0].set_title('(a) Original calexp', fontsize=13)

axes[1].imshow(calexp_injected, cmap='gray', origin='lower', vmin=v1, vmax=v2)
for info in calexp_info:
    c = plt.cm.plasma(np.clip((info['magnitude'] - MAG_RANGE[0]) / (MAG_RANGE[1] - MAG_RANGE[0]), 0, 1))
    axes[1].add_patch(Circle((info['x'], info['y']), info['r_half'] * 2,
                             fill=False, ec=c, lw=1.2, alpha=0.8))
axes[1].set_title(f'(b) After injection ({len(calexp_info)} clusters)', fontsize=13)

diff = calexp_injected.astype(float) - calexp_image.astype(float)
d_show = diff.copy()
d_show[d_show <= 0] = np.nan
vd = d_show[np.isfinite(d_show)]
if len(vd) > 0:
    im = axes[2].imshow(d_show, cmap='hot', origin='lower',
                        norm=LogNorm(vmin=max(0.1, np.nanpercentile(vd, 1)), vmax=np.nanmax(vd)))
    plt.colorbar(im, ax=axes[2], shrink=0.85, label='Injected flux')
axes[2].set_title('(c) Difference (injected only)', fontsize=13)

for ax in axes:
    ax.set_xlabel('X (pixels)')
    ax.set_ylabel('Y (pixels)')

fig.suptitle(f'Calexp Injection — GalSim {PSF_TYPE.title()} PSF (FWHM={PSF_FWHM_ARCSEC}")', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('calexp_injection.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================================================
# PLOT: POSTAGE STAMPS
# ==============================================================================
sorted_info = sorted(calexp_info, key=lambda d: d['magnitude'])
n_stamps = min(10, len(sorted_info))
picked = [sorted_info[i] for i in np.linspace(0, len(sorted_info)-1, n_stamps, dtype=int)]

fig, axes = plt.subplots(3, n_stamps, figsize=(2.4 * n_stamps, 7))
if n_stamps == 1:
    axes = axes.reshape(3, 1)

for col, info in enumerate(picked):
    cx, cy = int(info['x']), int(info['y'])
    r = max(int(info['r_half'] * 4), 30)
    y0, y1 = max(0, cy-r), min(ny_cal, cy+r)
    x0, x1 = max(0, cx-r), min(nx_cal, cx+r)
    
    s_orig = calexp_image[y0:y1, x0:x1].astype(float)
    s_inj = calexp_injected[y0:y1, x0:x1].astype(float)
    s_diff = s_inj - s_orig
    vp1, vp2 = np.percentile(s_inj, [0.5, 99.5])
    
    axes[0, col].imshow(s_orig, cmap='gray', origin='lower', vmin=vp1, vmax=vp2)
    axes[0, col].set_title(f'mag={info["magnitude"]:.1f}', fontsize=8)
    axes[1, col].imshow(s_inj, cmap='gray', origin='lower', vmin=vp1, vmax=vp2)
    axes[1, col].set_title(f'r½={info["r_half"]:.1f}', fontsize=8)
    
    dp = np.clip(s_diff, 0, None)
    if dp.max() > 0:
        axes[2, col].imshow(dp, cmap='inferno', origin='lower',
                            norm=LogNorm(vmin=max(dp[dp>0].min(), 0.01), vmax=dp.max()))
    axes[2, col].set_title(info['profile_type'], fontsize=8)
    
    for row in range(3):
        axes[row, col].plot(cx-x0, cy-y0, 'r+', ms=7, mew=1)
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])

axes[0, 0].set_ylabel('Before', fontsize=11)
axes[1, 0].set_ylabel('After', fontsize=11)
axes[2, 0].set_ylabel('Cluster only', fontsize=11)
fig.suptitle('Calexp — Postage Stamps (bright → faint)', fontsize=13)
plt.tight_layout()
plt.savefig('calexp_stamps.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part B: Inject into deepCoadd

In [ ]:
# ==============================================================================
# LOAD DEEPCOADD
# ==============================================================================
coadd_dataId = {'tract': 3828, 'patch': 24, 'band': 'i'}
deepCoadd = butler.get('deepCoadd', dataId=coadd_dataId)

coadd_image = deepCoadd.image.array.copy()
coadd_ny, coadd_nx = coadd_image.shape

print(f'deepCoadd: {coadd_dataId}')
print(f'Shape: {coadd_image.shape}')

In [ ]:
# ==============================================================================
# CREATE GALSIM PSF FOR COADD
# ==============================================================================
coadd_psf = make_galsim_psf(PSF_FWHM_ARCSEC, COADD_PIXEL_SCALE, psf_type=PSF_TYPE, beta=PSF_BETA)
coadd_psf_fwhm_px = PSF_FWHM_ARCSEC / COADD_PIXEL_SCALE

print(f'GalSim PSF for coadd: {PSF_TYPE}')
print(f'  FWHM: {PSF_FWHM_ARCSEC}" = {coadd_psf_fwhm_px:.2f} px')

In [ ]:
# ==============================================================================
# GENERATE CATALOG AND INJECT INTO COADD
# ==============================================================================
coadd_catalog = make_cluster_catalog(
    n_clusters=40,
    image_shape=coadd_image.shape,
    mag_range=(19.0, 26.0),
    r_half_range=(2.0, 30.0),
    zeropoint=ZEROPOINT,
    edge_buffer=300,
    seed=123,
)

print(f'Catalog: {len(coadd_catalog)} clusters')
print('Injecting...')

coadd_injected, coadd_info = inject_catalog(
    coadd_image, coadd_catalog, coadd_psf, COADD_PIXEL_SCALE,
    add_poisson=True, seed=123
)

print(f'Successfully injected {len(coadd_info)} clusters')

In [ ]:
# ==============================================================================
# PLOT: DEEPCOADD BEFORE / AFTER / DIFFERENCE
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
cv1, cv2 = np.percentile(coadd_image, [1, 99])

axes[0].imshow(coadd_image, cmap='gray', origin='lower', vmin=cv1, vmax=cv2)
axes[0].set_title('(a) Original deepCoadd', fontsize=13)

axes[1].imshow(coadd_injected, cmap='gray', origin='lower', vmin=cv1, vmax=cv2)
for info in coadd_info:
    c = plt.cm.plasma(np.clip((info['magnitude'] - 19) / (26 - 19), 0, 1))
    axes[1].add_patch(Circle((info['x'], info['y']), info['r_half'] * 2,
                             fill=False, ec=c, lw=1.2, alpha=0.8))
axes[1].set_title(f'(b) After injection ({len(coadd_info)} clusters)', fontsize=13)

cd = coadd_injected.astype(float) - coadd_image.astype(float)
cd[cd <= 0] = np.nan
cdv = cd[np.isfinite(cd)]
if len(cdv) > 0:
    im = axes[2].imshow(cd, cmap='hot', origin='lower',
                        norm=LogNorm(vmin=max(0.1, np.nanpercentile(cdv, 1)), vmax=np.nanmax(cdv)))
    plt.colorbar(im, ax=axes[2], shrink=0.85, label='Injected flux')
axes[2].set_title('(c) Difference (injected only)', fontsize=13)

for ax in axes:
    ax.set_xlabel('X (pixels)')
    ax.set_ylabel('Y (pixels)')

fig.suptitle(f'deepCoadd Injection — GalSim {PSF_TYPE.title()} PSF (FWHM={PSF_FWHM_ARCSEC}")', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('coadd_injection.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary

In [ ]:
# ==============================================================================
# SUMMARY
# ==============================================================================
print('='*60)
print('INJECTION SUMMARY')
print('='*60)
print(f'Profile Source: src.light_profiles (Plummer/King/EFF/Sersic)')
print(f'PSF: GalSim {PSF_TYPE.title()}, FWHM={PSF_FWHM_ARCSEC}"')
print()
print('Calexp:')
print(f'  Shape: {calexp_image.shape}')
print(f'  Pixel scale: {calexp_pixel_scale:.4f}"/px')
print(f'  Injected: {len(calexp_info)} clusters')
print()
print('deepCoadd:')
print(f'  Shape: {coadd_image.shape}')
print(f'  Pixel scale: {COADD_PIXEL_SCALE}"/px')
print(f'  Injected: {len(coadd_info)} clusters')
print()
print('Saved figures:')
import glob
for f in sorted(glob.glob('*.png')):
    print(f'  {f}')